In [ ]:
from libraries import *
from parameters import *
from sklearn.cross_decomposition import PLSCanonical, PLSRegression, CCA
from sklearn.model_selection import cross_val_score
from numpy import asarray
from numpy import savetxt
from adjustText import adjust_text
import seaborn as sns
from sklearn.cluster import SpectralBiclustering
%matplotlib inline
import umap

In [ ]:
%load_ext rpy2.ipython

In [ ]:
os.getcwd()
os.chdir(projectDir)

In [ ]:
coefsAll = pd.read_csv("./MixedEffectLMOutputs/ME_SignificantBetaCoefs.csv", header=0, index_col=0)
coefsAll.index = [cond.split('_')[1] for cond in coefsAll.index]

In [ ]:
guideModules = pd.read_csv("./TextFiles/ME_GuideModules_leiden_6_Modules.csv", 
                           header=0, index_col=0)
guideModules = guideModules.loc[coefsAll.index]

Generate guide modules with leiden

In [ ]:
koGuidesAnnDat = sc.AnnData(X=coefsAll)
koGuidesAnnDat.obs["GuideGroup"] = ["K"+str(x) for x in guideModules.GuideGroup]
sc.pp.pca(koGuidesAnnDat, n_comps=50, svd_solver='arpack')
koGuidesAnnDat.obs["KOGenes"] = coefsAll.index
#sc.pp.highly_variable_genes(koGuidesAnnDat, n_top_genes=200)
sc.pp.neighbors(koGuidesAnnDat,  n_neighbors=15, metric="euclidean")
#sc.tl.leiden(koGuidesAnnDat, resolution=0.64)
sc.tl.umap(koGuidesAnnDat)

f, ax = plt.subplots(figsize=(6, 6))
sc.pl.umap(koGuidesAnnDat, color='GuideGroup',ax=ax, size=100, 
           legend_fontoutline=3, legend_fontsize=14,legend_loc='right margin',
           legend_fontweight='normal', save="Figure_2A.pdf")

In [ ]:
sc.tl.dendrogram(koGuidesAnnDat, groupby='GuideGroup')
sc.pl.dendrogram(koGuidesAnnDat, groupby='GuideGroup')

In [ ]:
nClust=len(koGuidesAnnDat.obs["GuideGroup"].unique())
guideModules = koGuidesAnnDat.obs[["KOGenes", "GuideGroup"]]
guideModules.columns = ["GuideName", "GuideGroup"]
guideModules

In [ ]:
guideModules = guideModules.sort_values(["GuideGroup", "GuideName"], ascending = (True, True))

In [ ]:
guideModulesNew = pd.concat([guideModules.loc[guideModules.GuideGroup=="K4",], guideModules.loc[guideModules.GuideGroup=="K0",]])
guideModulesNew = pd.concat([guideModulesNew, guideModules.loc[guideModules.GuideGroup=="K1",]])
guideModulesNew = pd.concat([guideModulesNew, guideModules.loc[guideModules.GuideGroup=="K5",]])
guideModulesNew = pd.concat([guideModulesNew, guideModules.loc[guideModules.GuideGroup=="K3",]])
guideModulesNew = pd.concat([guideModulesNew, guideModules.loc[guideModules.GuideGroup=="K2",]])


guideModules = guideModulesNew

In [ ]:
guideCov = pd.DataFrame(np.corrcoef(coefsAll), index=coefsAll.index, columns=coefsAll.index)
guideCov = guideCov.loc[guideModules.GuideName,guideModules.GuideName]

In [ ]:
guideModules["GuideColor"] = ""
guideModules.loc[guideModules["GuideGroup"] == "K0", "GuideColor"] = koGuidesAnnDat.uns["GuideGroup_colors"][0]
guideModules.loc[guideModules["GuideGroup"] == "K1", "GuideColor"] = koGuidesAnnDat.uns["GuideGroup_colors"][1]
guideModules.loc[guideModules["GuideGroup"] == "K2", "GuideColor"] = koGuidesAnnDat.uns["GuideGroup_colors"][2]
guideModules.loc[guideModules["GuideGroup"] == "K3", "GuideColor"] = koGuidesAnnDat.uns["GuideGroup_colors"][3]
guideModules.loc[guideModules["GuideGroup"] == "K4", "GuideColor"] = koGuidesAnnDat.uns["GuideGroup_colors"][4]
guideModules.loc[guideModules["GuideGroup"] == "K5", "GuideColor"] = koGuidesAnnDat.uns["GuideGroup_colors"][5]

In [ ]:
sns.clustermap(guideCov, row_cluster=False, col_cluster= False,
               cmap=plt.cm.PRGn, vmin=-0.5, vmax=0.5, row_colors=guideModules.GuideColor,
               col_colors=guideModules.GuideColor)
plt.savefig('Figure_2C_1.pdf', 
           dpi=300)

In [ ]:
# geneModules = pd.read_csv("./TextFiles/SubClustered_GeneModules.csv", 
#                            header=0, index_col=None)

# geneModules.loc[geneModules.GeneGroup == "G7", "GeneGroup"] = "G10"
# geneModules.loc[geneModules.GeneGroup == "G6", "GeneGroup"] = "G9"
# geneModules.loc[geneModules.GeneGroup == "G5", "GeneGroup"] = "G8"
# geneModules.loc[geneModules.GeneGroup == "G4", "GeneGroup"] = "G7"
# geneModules.loc[geneModules.GeneGroup == "G3", "GeneGroup"] = "G6"
# geneModules.loc[geneModules.GeneGroup == "G2_1", "GeneGroup"] = "G5"
# geneModules.loc[geneModules.GeneGroup == "G2_0", "GeneGroup"] = "G4"
# geneModules.loc[geneModules.GeneGroup == "G1_1", "GeneGroup"] = "G3"
# geneModules.loc[geneModules.GeneGroup == "G1_0", "GeneGroup"] = "G2"
# geneModules.loc[geneModules.GeneGroup == "G0_1", "GeneGroup"] = "G1"
# geneModules.loc[geneModules.GeneGroup == "G0_0", "GeneGroup"] = "G0"

# geneModules.GeneColor = '#A6CEE3'

# geneModules.loc[geneModules.GeneGroup == "G0", "GeneColor"] = '#A6CEE3'
# geneModules.loc[geneModules.GeneGroup == "G1", "GeneColor"] = '#1F78B4'
# geneModules.loc[geneModules.GeneGroup == "G2", "GeneColor"] = '#B2DF8A'
# geneModules.loc[geneModules.GeneGroup == "G3", "GeneColor"] = '#33A02C'
# geneModules.loc[geneModules.GeneGroup == "G4", "GeneColor"] = '#FB9A99'
# geneModules.loc[geneModules.GeneGroup == "G5", "GeneColor"] = '#FDBF6F'
# geneModules.loc[geneModules.GeneGroup == "G6", "GeneColor"] = '#FF7F00'
# geneModules.loc[geneModules.GeneGroup == "G7", "GeneColor"] = '#CAB2D6'
# geneModules.loc[geneModules.GeneGroup == "G8", "GeneColor"] = '#6A3D9A'
# geneModules.loc[geneModules.GeneGroup == "G9", "GeneColor"] = '#FFFF99'
# geneModules.loc[geneModules.GeneGroup == "G10", "GeneColor"] = "#B5651D"


# #geneModules = geneModules.loc[coefsAll.columns]
# geneModules

# geneModules.to_csv("./TextFiles/ME_GeneModules_leiden_11_Modules.csv", index=False)

In [ ]:
geneModules = pd.read_csv("./TextFiles/ME_GeneModules_leiden_11_Modules.csv", 
                           header=0, index_col=None)
geneModules.index= geneModules.GeneName	
geneModules = geneModules.loc[coefsAll.columns]

geneModules

Generate gene modules with leiden

In [ ]:
koGenesAnnDat = sc.AnnData(X=coefsAll.transpose())
koGenesAnnDat.obs["GeneGroup"] = [str(x) for x in geneModules.GeneGroup]
sc.pp.pca(koGenesAnnDat, n_comps=50, svd_solver='arpack')
sc.pp.neighbors(koGenesAnnDat, n_neighbors=4, metric="euclidean")
#sc.tl.leiden(koGenesAnnDat, resolution=0.8)
sc.tl.umap(koGenesAnnDat)
koGenesAnnDat.obs["EffectedGenes"] = coefsAll.columns

f, ax = plt.subplots(figsize=(8, 8))
sc.pl.umap(koGenesAnnDat, color='GeneGroup',ax=ax, size=100, palette = 'Paired', 
           legend_fontoutline=3, 
           legend_fontsize=14,legend_loc='right margin',
           legend_fontweight='normal',
          save="Figure_2B_3.pdf")

In [ ]:
#geneModulesNew = pd.concat([geneModules.loc[geneModules.GeneGroup.isin(["0","1","2","4","6","7"]),], geneModules.loc[geneModules.GeneGroup.isin(["3","5"]),]])

geneModules = pd.DataFrame(pd.read_csv("./TextFiles/ME_GeneModules_leiden_11_Modules.csv", 
                           header=0, index_col=None))

geneModulesNew = pd.concat([geneModules.loc[geneModules.GeneGroup=="G6",], geneModules.loc[geneModules.GeneGroup=="G9",]])
geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GeneGroup=="G10",]])

geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GeneGroup=="G5",]])
geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GeneGroup=="G4",]])
geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GeneGroup=="G7",]])
geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GeneGroup=="G0",]])
geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GeneGroup=="G1",]])

geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GeneGroup=="G3",]])
geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GeneGroup=="G2",]])
geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GeneGroup=="G8",]])


geneModulesNew


In [ ]:
geneModulesNew.to_csv("./TextFiles/ME_GeneModules_leiden_11_Modules.csv", index=False)

In [ ]:
geneCov = pd.DataFrame(np.corrcoef(coefsAll.transpose()), index=coefsAll.columns, columns=coefsAll.columns)
geneCov = geneCov.loc[geneModules.GeneName,geneModules.GeneName]

In [ ]:
sns.clustermap(geneCov, row_cluster=False, col_cluster= False, cmap=plt.cm.PRGn,
               vmin=-0.5, vmax=0.5, 
               row_colors=list(geneModules.GeneColor), 
               col_colors=list(geneModules.GeneColor))

plt.savefig('Figure_2C_2.pdf', 
           dpi=300)


In [ ]:
geneModules.GeneColor

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
ax.matshow(coefsAll.loc[guideCov.index, geneCov.index], cmap=plt.cm.coolwarm, vmin=-0.2, vmax=0.2)


In [ ]:
sns.clustermap(coefsAll.loc[guideCov.index, geneCov.index], 
               row_cluster=False, col_cluster= False, 
               cmap=plt.cm.bwr, vmin=-0.2, vmax=0.2, figsize=(18,14))
plt.savefig('Figure_2C_3.pdf', 
           dpi=300)
